# Heme SH → ChatGPT upload zips (lazy)

**Goal:** run this once → get a folder of zips on Drive → attach one zip per ChatGPT chat.

## What you get

For each lecture series (AML, BM Intro, MDS/MPN, …):

`series_<slug>_chatGPT_upload.zip`

Each zip is self-contained:
- TNK style exemplar + SOP + WHO JSON (if found on Drive / GCS)
- COMMON + series handoff + paste-ready `CHATGPT_PROMPT.txt`
- lecture ZIP(s) and/or sidecars (`manifest`, `frames.jsonl`, `segments.jsonl`)

**TNK** = finished T/NK-lymphomas Anki **exemplar** (style law + accepted tags). Not a lecture to convert.

## How to use (3 steps)

1. Put these somewhere under MyDrive **once** (any folder):
   - `Heme_SH_TNK_Lymphomas_Contextual_Cloze_Final_Package.zip`
   - `WHO_WHO_JSON_PROCESSED_HEME.json`
2. Edit **Cell 2** if you want (`SERIES_CHOICE = "ALL"` is fine).
3. **Runtime → Run all.** Then open Drive → `Heme_Anki_ChatGPT_Zips/` and upload zips to ChatGPT.

Inside each zip, open `CHATGPT_PROMPT.txt` and paste that into the chat after attaching the zip.

In [ ]:
# Cell 1 — auth
!pip -q install google-cloud-storage

from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")

from google.cloud import storage
from pathlib import Path
import json, shutil, zipfile, textwrap
from datetime import datetime, timezone

PROJECT = "pathology-annotation-project"
HUB = "pathology_hub"
BUILDER = "02_normalized/anki/heme_sh_contextual_cloze_builder_v0_1"
DECK = "02_normalized/lectures/deck_packages"

client = storage.Client(project=PROJECT)
hub = client.bucket(HUB)

def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print("ok", utc_now())

In [ ]:
# Cell 2 — knobs (lazy defaults already set)

OUT_ROOT = Path("/content/drive/MyDrive/Heme_Anki_ChatGPT_Zips")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# "ALL" or one slug: bm_intro, mds_mpn, aml, spleen, ...
SERIES_CHOICE = "ALL"

# "full" = include lecture package ZIP(s) (bigger, has frames/ + transcript files)
# "light" = skip lecture ZIPs; use sidecars only (smaller; usually enough for ChatGPT)
MODE = "light"

DOWNLOAD_SEGMENTS = True   # useful; set False if a series zip is too big
DOWNLOAD_CHUNKS = True     # optional nav index — never use as syllabus
UPLOAD_TNK_WHO_TO_GCS = False  # True = also land TNK/WHO in project GCS shared/

# Only needed if auto-find fails:
MANUAL_TNK = None  # Path("/content/drive/MyDrive/.../Heme_SH_TNK_Lymphomas_Contextual_Cloze_Final_Package.zip")
MANUAL_WHO = None  # Path("/content/drive/MyDrive/.../WHO_WHO_JSON_PROCESSED_HEME.json")

print("OUT_ROOT =", OUT_ROOT)
print("SERIES_CHOICE =", SERIES_CHOICE, "MODE =", MODE)

In [ ]:
# Cell 3 — catalog
SERIES = {
    "aggressive_b_cell": {
        "title": "Aggressive B-Cell",
        "handoff": "HANDOFF_AGGRESSIVE_B_CELL_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_aggressive_b_cell_v0_1"],
        "zips": ["Heme_SH_Aggressive_B_Cell_chatgpt_readable_package.zip"],
    },
    "aml": {
        "title": "AML",
        "handoff": "HANDOFF_AML_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_aml_v0_1"],
        "zips": ["Heme_SH_AML_package.zip"],
    },
    "bm_failure_syndromes": {
        "title": "BM Failure Syndromes",
        "handoff": "HANDOFF_BM_FAILURE_SYNDROMES_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_bm_failure_syndromes_v0_1"],
        "zips": ["Heme_SH_BM_Failure_Syndromes_package.zip"],
    },
    "bm_intro": {
        "title": "BM Intro",
        "handoff": "HANDOFF_BM_INTRO_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_bm_intro_v0_1"],
        "zips": ["Heme_SH_BM_Intro_package.zip"],
        "high_yield_survey": True,
    },
    "bm_systemic_manifestations": {
        "title": "BM Systemic Manifestations",
        "handoff": "HANDOFF_BM_SYSTEMIC_MANIFESTATIONS_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_bm_systemic_manifestations_v0_1"],
        "zips": ["Heme_SH_BM_Systemic_Manifestations_package.zip"],
    },
    "histiocytic": {
        "title": "Histiocytic",
        "handoff": "HANDOFF_HISTIOCYTIC_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_histiocytic_v0_1"],
        "zips": ["Heme_SH_Histiocytic_package.zip"],
    },
    "hodgkin_nlp": {
        "title": "Hodgkin NLP",
        "handoff": "HANDOFF_HODGKIN_NLP_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_hodgkin_nlp_v0_1"],
        "zips": ["Heme_SH_Hodgkin_NLP_package.zip"],
    },
    "hodgkin_overview": {
        "title": "Hodgkin Overview",
        "handoff": "HANDOFF_HODGKIN_OVERVIEW_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_hodgkin_overview_v0_1"],
        "zips": ["Heme_SH_Hodgkin_Overview_package.zip"],
    },
    "hodgkin_t_nk_cell": {
        "title": "Hodgkin T/NK-Cell",
        "handoff": "HANDOFF_HODGKIN_T_NK_CELL_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_hodgkin_t_nk_cell_1_v0_1", "heme_sh_hodgkin_t_nk_cell_2_v0_1"],
        "zips": [
            "Heme_SH_Hodgkin_T_NK_Cell_1_package.zip",
            "Heme_SH_Hodgkin_T_NK_Cell_2_package.zip",
        ],
    },
    "ia_lpd": {
        "title": "IA-LPD",
        "handoff": "HANDOFF_IA_LPD_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_ia_lpd_v0_1"],
        "zips": ["Heme_SH_IA_LPD_package.zip"],
    },
    "ihc_for_lpd": {
        "title": "IHC for LPD",
        "handoff": "HANDOFF_IHC_FOR_LPD_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_ihc_for_lpd_v0_1"],
        "zips": ["Heme_SH_IHC_for_LPD_package.zip"],
    },
    "mds_mpn": {
        "title": "MDS/MPN",
        "handoff": "HANDOFF_MDS_MPN_ANKI_BUILDER_v0_1.md",
        "packages": [
            "heme_sh_mds_mpn_1_v0_1",
            "heme_sh_mds_mpn_2_v0_1",
            "heme_sh_mds_mpn_3_v0_1",
        ],
        "zips": [
            "Heme_SH_MDS_MPN_1_package.zip",
            "Heme_SH_MDS_MPN_2_package.zip",
            "Heme_SH_MDS_MPN_3_package.zip",
        ],
    },
    "plasma_cell": {
        "title": "Plasma Cell",
        "handoff": "HANDOFF_PLASMA_CELL_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_plasma_cell_v0_1"],
        "zips": ["Heme_SH_Plasma_Cell_package.zip"],
    },
    "pt_lpd": {
        "title": "PT-LPD",
        "handoff": "HANDOFF_PT_LPD_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_pt_lpd_v0_1"],
        "zips": ["Heme_SH_PT_LPD_package.zip"],
    },
    "reactive_lymphoid_hyperplasia": {
        "title": "Reactive Lymphoid Hyperplasia",
        "handoff": "HANDOFF_REACTIVE_LYMPHOID_HYPERPLASIA_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_reactive_lymphoid_hyperplasia_v0_1"],
        "zips": ["Heme_SH_Reactive_Lymphoid_Hyperplasia_package.zip"],
    },
    "small_b_cell": {
        "title": "Small B-Cell",
        "handoff": "HANDOFF_SMALL_B_CELL_ANKI_BUILDER_v0_1.md",
        "packages": [
            "heme_sh_small_b_cell_1_of_2_v0_1",
            "heme_sh_small_b_cell_2_of_2_v0_1",
        ],
        "zips": [
            "Heme_SH_Small_B_Cell_1_of_2_package.zip",
            "Heme_SH_Small_B_Cell_2_of_2_package.zip",
        ],
    },
    "spleen": {
        "title": "Spleen",
        "handoff": "HANDOFF_SPLEEN_ANKI_BUILDER_v0_1.md",
        "packages": ["heme_sh_spleen_v0_1"],
        "zips": ["Heme_SH_Spleen_package.zip"],
    },
}
print(len(SERIES), "series ready")

In [ ]:
# Cell 4 — helpers + find TNK / WHO (fast paths first)

TNK_NAME = "Heme_SH_TNK_Lymphomas_Contextual_Cloze_Final_Package.zip"
WHO_NAME = "WHO_WHO_JSON_PROCESSED_HEME.json"

def gcs_get(name: str, dest: Path) -> bool:
    blob = hub.blob(name)
    if not blob.exists():
        print("  missing gs://" + HUB + "/" + name)
        return False
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size == blob.size:
        return True
    blob.download_to_filename(str(dest))
    print("  got", dest.name, f"({dest.stat().st_size:,} B)")
    return True

def find_file(filename: str):
    # Prefer likely folders, then limited walk
    roots = [
        Path("/content/drive/MyDrive/3-Resources"),
        Path("/content/drive/MyDrive/3 - Resources"),
        Path("/content/drive/MyDrive/Heme"),
        Path("/content/drive/MyDrive/Anki"),
        Path("/content/drive/MyDrive"),
    ]
    for root in roots:
        if not root.exists():
            continue
        # shallow: root and one level down first
        direct = root / filename
        if direct.is_file():
            return direct
        try:
            for child in root.iterdir():
                if child.is_dir():
                    cand = child / filename
                    if cand.is_file():
                        return cand
        except Exception:
            pass
    # broader but still capped
    root = Path("/content/drive/MyDrive")
    if root.exists():
        n = 0
        for p in root.rglob(filename):
            if p.is_file():
                return p
            n += 1
            if n > 50000:
                break
    return None

tnk = Path(MANUAL_TNK) if MANUAL_TNK else find_file(TNK_NAME)
who = Path(MANUAL_WHO) if MANUAL_WHO else find_file(WHO_NAME)

# also try GCS shared/ if user already uploaded
cache = Path("/content/anki_shared_cache")
cache.mkdir(exist_ok=True)
if tnk is None:
    if gcs_get(f"{BUILDER}/shared/{TNK_NAME}", cache / TNK_NAME):
        tnk = cache / TNK_NAME
if who is None:
    if gcs_get(f"{BUILDER}/shared/{WHO_NAME}", cache / WHO_NAME):
        who = cache / WHO_NAME

print("TNK:", tnk)
print("WHO:", who)
if tnk is None:
    print("⚠️ Put TNK zip on Drive (any folder) or set MANUAL_TNK — style/tags will be incomplete.")
if who is None:
    print("⚠️ Put WHO JSON on Drive or set MANUAL_WHO — shared backs will be weaker.")

In [ ]:
# Cell 5 — build ALL (or one) ChatGPT zip(s)

WORK = Path("/content/anki_zip_work")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir()

SHARED_STAGING = WORK / "_shared"
SHARED_STAGING.mkdir()

# shared authority
gcs_get(f"{BUILDER}/shared/Pathology_Anki_Contextual_Cloze_SOP_v1_1.pdf",
        SHARED_STAGING / "Pathology_Anki_Contextual_Cloze_SOP_v1_1.pdf")
for doc in [
    "HANDOFF_HEME_SH_ANKI_BUILDER_COMMON_v0_1.md",
    "HANDOFF_HEME_SH_ANKI_BUILDER_INDEX_v0_1.md",
    "HANDOFF_CHATGPT_HEME_ANKI_PROMPTS_v0_1.md",
]:
    gcs_get(f"{BUILDER}/docs/{doc}", SHARED_STAGING / doc)

if tnk and tnk.exists():
    shutil.copy2(tnk, SHARED_STAGING / TNK_NAME)
if who and who.exists():
    shutil.copy2(who, SHARED_STAGING / WHO_NAME)

if UPLOAD_TNK_WHO_TO_GCS:
    for fname in [TNK_NAME, WHO_NAME]:
        lp = SHARED_STAGING / fname
        if lp.exists():
            hub.blob(f"{BUILDER}/shared/{fname}").upload_from_filename(str(lp))
            print("uploaded", fname, "→ GCS shared/")


def write_prompt(path: Path, slug: str, title: str, survey: bool):
    survey_bit = ""
    if survey:
        survey_bit = textwrap.dedent(
            '''
            CRITICAL FOR THIS SERIES:
            - chunks_indexable may be empty/sparse — IGNORE for scope.
            - This talk is HIGH YIELD. Scope from lecture_index/transcript/frames/segments.
            - Use lecture-aligned shared backs for methods/approach pearls when no WHO disease applies.
            - Never invent tags; ask if no accepted tag fits.
            '''
        )
    text = textwrap.dedent(
        f'''
        You are building a Pathology Anki contextual-cloze deck.

        Attached: this zip for series "{title}" ({slug}).

        Read FIRST inside the zip:
        1) docs/HANDOFF_HEME_SH_ANKI_BUILDER_COMMON_v0_1.md
        2) docs/{SERIES[slug]["handoff"]}
        3) style/Pathology_Anki_Contextual_Cloze_SOP_v1_1.pdf
        4) style/{TNK_NAME}  (TNK = style exemplar + accepted tags ONLY)

        Hard rules:
        - Tag authority = accepted-tag JSON inside the TNK exemplar ONLY.
        - Syllabus = transcript / lecture_index / frames (NOT chunks_indexable).
        - Do not use tag_audit, chunk_audit, audit.json, or frame primary_tag for tags.
        - Diagnosis visible+bold; one-token clozes; Tier-1 images; byte-identical shared back per tag.
        - Prefer a smaller coherent deck over a mechanical dump.

        {survey_bit}

        First output ONLY: a proposed card inventory table
        (topic | teaching pearl | candidate PrimaryTag | frame timestamp | why high-yield).
        Wait for my approval before writing all cards.

        Deliverables after approval: APKG (or buildable notes), QA CSV, card inventory CSV,
        shared_backs.json, notes on builder steps.
        '''
    ).strip() + "\n"
    path.write_text(text)


def zip_folder(src: Path, dest_zip: Path):
    if dest_zip.exists():
        dest_zip.unlink()
    with zipfile.ZipFile(dest_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in src.rglob("*"):
            if p.is_file():
                zf.write(p, p.relative_to(src).as_posix())


slugs = list(SERIES) if SERIES_CHOICE.upper() == "ALL" else [SERIES_CHOICE]
assert all(s in SERIES for s in slugs), slugs

sidecar_names = ["manifest.json", "frames.jsonl"]
if DOWNLOAD_SEGMENTS:
    sidecar_names.append("segments.jsonl")
if DOWNLOAD_CHUNKS:
    sidecar_names.append("chunks_indexable.jsonl")

made = []
for slug in slugs:
    meta = SERIES[slug]
    bundle = WORK / slug
    if bundle.exists():
        shutil.rmtree(bundle)
    (bundle / "style").mkdir(parents=True)
    (bundle / "docs").mkdir()
    (bundle / "who").mkdir()

    # style / who / docs
    for fname in [TNK_NAME, "Pathology_Anki_Contextual_Cloze_SOP_v1_1.pdf"]:
        src = SHARED_STAGING / fname
        if src.exists():
            shutil.copy2(src, bundle / "style" / fname)
    if (SHARED_STAGING / WHO_NAME).exists():
        shutil.copy2(SHARED_STAGING / WHO_NAME, bundle / "who" / WHO_NAME)

    for doc in [
        "HANDOFF_HEME_SH_ANKI_BUILDER_COMMON_v0_1.md",
        "HANDOFF_CHATGPT_HEME_ANKI_PROMPTS_v0_1.md",
        meta["handoff"],
    ]:
        if doc == meta["handoff"]:
            gcs_get(f"{BUILDER}/docs/{doc}", bundle / "docs" / doc)
        else:
            src = SHARED_STAGING / doc
            if src.exists():
                shutil.copy2(src, bundle / "docs" / doc)
            else:
                gcs_get(f"{BUILDER}/docs/{doc}", bundle / "docs" / doc)

    write_prompt(
        bundle / "CHATGPT_PROMPT.txt",
        slug,
        meta["title"],
        bool(meta.get("high_yield_survey")),
    )

    for i, (pkg, zname) in enumerate(zip(meta["packages"], meta["zips"]), start=1):
        lec = bundle / f"lecture_{i}"
        lec.mkdir(exist_ok=True)
        if MODE == "full":
            gcs_get(zname, lec / zname)
        for sc in sidecar_names:
            gcs_get(f"{DECK}/{pkg}/{sc}", lec / sc)

    (bundle / "README.txt").write_text(
        f"Series: {meta['title']} ({slug})\n"
        f"Mode: {MODE}\n"
        "1) Attach this whole zip in ChatGPT.\n"
        "2) Paste CHATGPT_PROMPT.txt into the chat.\n"
        "3) TNK zip inside style/ is STYLE ONLY — accepted tags live there.\n"
        "4) Never use tag_audit/chunk_audit/audit.json.\n"
    )

    out_zip = OUT_ROOT / f"series_{slug}_chatGPT_upload.zip"
    zip_folder(bundle, out_zip)
    mb = out_zip.stat().st_size / (1024 * 1024)
    made.append((slug, out_zip, mb))
    print(f"✅ {slug}: {out_zip.name} ({mb:.1f} MB)")

audit = {
    "schema_version": "heme_anki_chatgpt_zips.v0_1",
    "created_at_utc": utc_now(),
    "mode": MODE,
    "out_root": str(OUT_ROOT),
    "tnk_present": bool(tnk and Path(tnk).exists()),
    "who_present": bool(who and Path(who).exists()),
    "zips": [{"slug": s, "path": str(p), "mb": round(m, 2)} for s, p, m in made],
    "known_limitations": [
        "light mode omits lecture package ZIPs; use frames.jsonl image_url + segments for content",
        "accepted_tags.json lives inside TNK exemplar zip until extracted",
        "ChatGPT attachment size limits may require light mode or dropping segments",
    ],
}
audit_path = OUT_ROOT / f"_audit_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json"
audit_path.write_text(json.dumps(audit, indent=2))

print("\n==== DONE ====\nDrive folder:", OUT_ROOT)
print(f"Made {len(made)} zip(s). Audit: {audit_path.name}")
print("\nUpload one zip per ChatGPT chat, then paste that zip's CHATGPT_PROMPT.txt")
if not audit["tnk_present"] or not audit["who_present"]:
    print("\nStill missing:", [x for x, ok in [("TNK", audit["tnk_present"]), ("WHO", audit["who_present"])] if not ok])

## After it runs

Drive → `MyDrive/Heme_Anki_ChatGPT_Zips/`

| File | Use |
|------|-----|
| `series_bm_intro_chatGPT_upload.zip` | Attach for BM Intro deck |
| `series_mds_mpn_chatGPT_upload.zip` | Attach for MDS/MPN |
| … | one chat per zip |

If a zip is too big for ChatGPT: set `MODE = "light"` (default) and/or `DOWNLOAD_SEGMENTS = False`, re-run that series.

If you want slide JPEGs inside the zip too, set `MODE = "full"` (larger).